# Quick validation notebook for the plugin SDK

This notebook verifies that:

- the SDK can generate a plugin skeleton
- the generated plugin exposes the expected runtime contract
- the manager can discover real plugin manifests from the repo

Run the cells in order.

In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import time

import requests

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'plugin_framework').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'PROJECT_ROOT={PROJECT_ROOT}')

PROJECT_ROOT=/home/eloygarcia/Escritorio/WorkInProgress/Preprocessing


In [2]:
from plugin_framework.plugin_sdk import PluginProjectGenerator, PluginManifest

project_dir = PROJECT_ROOT / 'plugin_framework' / 'examples' / 'generated_plugin_test'
if project_dir.exists():
    shutil.rmtree(project_dir)
project_dir.mkdir(parents=True, exist_ok=True)

manifest = PluginProjectGenerator().generate(project_dir, 'generated', 'Generated sample plugin for test')
print('Manifest:', manifest)
print('Files created:', sorted(p.name for p in project_dir.iterdir()))

Manifest: PluginManifest(name='generated', version='0.1.0', description='Generated sample plugin for test', entrypoint='app:app', port=8000, gpu=True, docker_image='preprocessing:generated', base_image='preprocessing:notebook', tags=['mammography'])
Files created: ['Dockerfile', 'README.md', '__init__.py', 'api.py', 'config.yaml', 'plugin.json', 'plugin.yaml', 'postprocessing.py', 'predictor.py', 'preprocessing.py', 'requirements.txt']


In [3]:
# Load the generated manifest and check the YAML contract.
manifest = PluginManifest.from_yaml(project_dir / 'plugin.yaml')
print('Name:', manifest.name)
print('Port:', manifest.port)
print('GPU:', manifest.gpu)

Name: generated
Port: 8000
GPU: True


In [ ]:
proc = subprocess.Popen(
    [sys.executable, 'api.py'],
    cwd=str(project_dir),
    env={**os.environ, 'PYTHONPATH': str(PROJECT_ROOT)},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Wait a bit for the service to start.
for _ in range(50):
    time.sleep(0.2)
    try:
        response = requests.get('http://127.0.0.1:8000/health', timeout=1.5)
        print('HEALTH_STATUS:', response.status_code)
        print(response.json())
        break
    except Exception:
        continue
else:
    out = proc.stdout.read() if proc.stdout else ''
    print('Startup timed out. Service output:')
    print(out)
    proc.terminate()
    raise RuntimeError('Generated plugin did not start correctly')

HEALTH_STATUS: 200
{'status': 'ok', 'name': 'generated', 'version': '0.1.0', 'gpu': True}


In [5]:
# Stop the running process after the validation.
proc.terminate()
try:
    proc.wait(timeout=5)
except subprocess.TimeoutExpired:
    proc.kill()
print('Plugin process stopped cleanly.')

Plugin process stopped cleanly.


In [6]:
from plugin_framework.plugin_manager import PluginManager

root = PROJECT_ROOT / 'plugin_framework' / 'plugins'
manager = PluginManager(root)
manager.discover()
print('Discovered plugins:', sorted(manager.get_catalog().keys()))
for name, plugin in manager.get_catalog().items():
    print(name, plugin.metadata)

Discovered plugins: ['xai']
xai {'name': 'xai', 'version': '0.1', 'description': '', 'entrypoint': 'app:app', 'port': 8004, 'gpu': True, 'docker_image': 'preprocessing:xai', 'base_image': 'preprocessing:notebook', 'tags': []}


## Summary

If all cells above ran without errors, then the SDK generation flow, the FastAPI runtime contract, and the plugin manager discovery contract are working together as expected.